# 브랜드 자가진단 대시보드 - 데이터 준비

**목적:** 기존 입점 브랜드를 위한 SWOT 대시보드용 통합 데이터셋 생성  
**핵심:** ABSA 감성 분석(323K건) + base_score + SLI를 제품 수준으로 통합  

## 출력 파일
1. `v_product_aspect_summary.csv` — 제품 × Aspect 감성 집계
2. `v_category_aspect_benchmark.csv` — 카테고리 × Aspect 벤치마크
3. `v_brand_dashboard_master.csv` — Tableau용 통합 마스터 테이블

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 경로 설정
BASE = Path('/sessions/upbeat-quirky-goldberg/mnt/04_Why-pi')
DATA = BASE / '02_outputs' / '00_data' / 'csv' / 'final'
DASH_OUT = BASE / '02_outputs' / 'DashBoard' / '데이터_추출'

print('데이터 경로:', DATA)
print('출력 경로:', DASH_OUT)

## 1. 원본 데이터 로드

In [ ]:
# 리뷰 - Aspect 감성
review_aspects = pd.read_csv(DATA / 'review_aspects.csv')
print(f'review_aspects: {len(review_aspects):,}건, 컬럼: {list(review_aspects.columns)}')

# 리뷰 - 기본 (review_id → product_code 매핑)
reviews_core = pd.read_csv(DATA / 'reviews_core.csv',
                           usecols=['review_id', 'product_code', 'rating'])
print(f'reviews_core: {len(reviews_core):,}건')

# 제품 기본
products_core = pd.read_csv(DATA / 'products_core.csv',
                            usecols=['product_code', 'brand_id', 'name', 'price'])
print(f'products_core: {len(products_core):,}건')

# 제품 카테고리
products_cat = pd.read_csv(DATA / 'products_category.csv')
print(f'products_category: {len(products_cat):,}건')

# 제품 통계
products_stats = pd.read_csv(DATA / 'products_stats.csv')
print(f'products_stats: {len(products_stats):,}건')

# 브랜드
brands = pd.read_csv(DATA / 'brands.csv', encoding='utf-8')
# fallback encoding
if brands.columns[0] != 'brand_id':
    brands = pd.read_csv(DATA / 'brands.csv', encoding='euc-kr')
brands.columns = ['brand_id', 'brand_name']
print(f'brands: {len(brands):,}건')

# SLI 결과
sli = pd.read_csv(DATA / 'sli_results.csv',
                  usecols=['product_code', 'final_soft_landing', 'ml_prob', 'confidence'])
print(f'sli_results: {len(sli):,}건')

# base_score (v_score_distribution에서 추출)
score_path = DATA / 'v_score_distribution.csv'
if score_path.exists():
    scores = pd.read_csv(score_path)
    print(f'v_score_distribution: {len(scores):,}건')
else:
    print('⚠ v_score_distribution.csv 미존재 — BQ에서 추출 필요')
    scores = None

## 2. Step 1: 제품별 Aspect 감성 집계

review_aspects → reviews_core JOIN → product_code × aspect 기준 집계

In [ ]:
# 미분류 aspect 제외 (8개 운영 aspect만)
VALID_ASPECTS = [
    '사용감/성능', '색상/발색', '재구매', '가격/가성비',
    '배송/포장', '재질/냄새', '용량/휴대', '디자인'
]

aspects_filtered = review_aspects[review_aspects['aspect'].isin(VALID_ASPECTS)].copy()
print(f'유효 aspect 리뷰: {len(aspects_filtered):,}건 (전체 {len(review_aspects):,}건 중)')

# review_id → product_code 매핑
aspects_with_product = aspects_filtered.merge(
    reviews_core[['review_id', 'product_code']],
    on='review_id',
    how='inner'
)
print(f'제품 매핑 완료: {len(aspects_with_product):,}건')

# 제품별 전체 리뷰 수 (aspect 무관)
product_total_reviews = reviews_core.groupby('product_code')['review_id'].nunique().reset_index()
product_total_reviews.columns = ['product_code', 'total_reviews']

In [ ]:
# 제품 × Aspect 집계
product_aspect = aspects_with_product.groupby(
    ['product_code', 'aspect']
).agg(
    mention_count=('review_id', 'nunique'),
    pos_count=('aspect_sentiment', lambda x: (x == 'positive').sum()),
    neu_count=('aspect_sentiment', lambda x: (x == 'neutral').sum()),
    neg_count=('aspect_sentiment', lambda x: (x == 'negative').sum()),
    avg_confidence=('aspect_confidence', 'mean')
).reset_index()

# total_reviews 합산
product_aspect = product_aspect.merge(product_total_reviews, on='product_code', how='left')

# 비율 계산
product_aspect['mention_rate'] = (product_aspect['mention_count'] / product_aspect['total_reviews'] * 100).round(2)
product_aspect['pos_pct'] = (product_aspect['pos_count'] / product_aspect['mention_count'] * 100).round(2)
product_aspect['neu_pct'] = (product_aspect['neu_count'] / product_aspect['mention_count'] * 100).round(2)
product_aspect['neg_pct'] = (product_aspect['neg_count'] / product_aspect['mention_count'] * 100).round(2)

print(f'\n제품 × Aspect 집계 결과: {len(product_aspect):,}행')
print(f'고유 제품 수: {product_aspect["product_code"].nunique()}')
print(f'\nAspect별 평균 긍정률:')
print(product_aspect.groupby('aspect')['pos_pct'].mean().sort_values(ascending=False).round(1))

# 저장
product_aspect.to_csv(DASH_OUT / 'v_product_aspect_summary.csv', index=False, encoding='utf-8-sig')
print(f'\n✅ v_product_aspect_summary.csv 저장 완료 ({len(product_aspect):,}행)')

## 3. Step 2: 카테고리 벤치마크 계산

In [ ]:
# 제품-카테고리 매핑 추가
pa_with_cat = product_aspect.merge(
    products_cat[['product_code', 'category_2']],
    on='product_code',
    how='left'
)

# 카테고리 × Aspect 벤치마크
cat_benchmark = pa_with_cat.groupby(
    ['category_2', 'aspect']
).agg(
    cat_avg_pos_pct=('pos_pct', 'mean'),
    cat_avg_neg_pct=('neg_pct', 'mean'),
    cat_avg_mention_rate=('mention_rate', 'mean'),
    cat_median_pos_pct=('pos_pct', 'median'),
    cat_product_count=('product_code', 'nunique')
).reset_index().round(2)

print(f'카테고리 벤치마크: {len(cat_benchmark):,}행')
print(f'고유 카테고리: {cat_benchmark["category_2"].nunique()}')
print(f'\n카테고리별 제품 수:')
print(cat_benchmark.groupby('category_2')['cat_product_count'].first().sort_values(ascending=False).head(10))

# 저장
cat_benchmark.to_csv(DASH_OUT / 'v_category_aspect_benchmark.csv', index=False, encoding='utf-8-sig')
print(f'\n✅ v_category_aspect_benchmark.csv 저장 완료 ({len(cat_benchmark):,}행)')

## 4. Step 3: 통합 마스터 테이블 생성

Tableau에서 사용할 단일 Long-format 마스터 테이블

In [ ]:
# ── 제품 프로필 생성 (기본정보 + 카테고리 + 통계 + 브랜드 + SLI) ──
product_profile = (
    products_core
    .merge(products_cat, on='product_code', how='left')
    .merge(brands, on='brand_id', how='left')
    .merge(products_stats[[
        'product_code', 'review_count', 'engagement_score',
        'cp_index', 'review_density'
    ]], on='product_code', how='left')
    .merge(sli[['product_code', 'final_soft_landing', 'ml_prob']], 
           on='product_code', how='left')
)

# base_score 통합 (있으면)
if scores is not None:
    score_cols = [c for c in scores.columns if c.startswith('score_') or c in ['product_code', 'base_score']]
    product_profile = product_profile.merge(
        scores[score_cols], on='product_code', how='left'
    )

# SLI 결측값 처리
product_profile['final_soft_landing'] = product_profile['final_soft_landing'].fillna(False)
product_profile['ml_prob'] = product_profile['ml_prob'].fillna(0)

print(f'제품 프로필: {len(product_profile):,}건')
print(f'컬럼: {list(product_profile.columns)}')

In [ ]:
# ── 마스터 테이블: 제품 프로필 × Aspect 감성 × 카테고리 벤치마크 ──

# product_aspect에 카테고리 정보 추가
master = product_aspect.merge(
    product_profile,
    on='product_code',
    how='inner'
)

# 카테고리 벤치마크 JOIN
master = master.merge(
    cat_benchmark,
    on=['category_2', 'aspect'],
    how='left'
)

# ── SWOT 파생변수 계산 ──

# 격차 (gap): 내 제품 vs 카테고리 평균
master['gap_pos'] = (master['pos_pct'] - master['cat_avg_pos_pct']).round(2)
master['gap_neg'] = (master['neg_pct'] - master['cat_avg_neg_pct']).round(2)

# SWOT 분류
def classify_swot(row):
    """aspect 단위 SWOT 분류"""
    if row['gap_pos'] >= 5:   # 긍정률이 카테고리 평균보다 5%p 이상 높으면 강점
        return 'Strength'
    elif row['gap_neg'] >= 3:  # 부정률이 카테고리 평균보다 3%p 이상 높으면 약점
        return 'Weakness'
    elif row['gap_pos'] >= 0:
        return 'Neutral_Good'
    else:
        return 'Neutral_Bad'

master['swot_label'] = master.apply(classify_swot, axis=1)

# 카테고리 내 base_score 순위 (있으면)
if 'base_score' in master.columns:
    master['cat_rank'] = master.groupby('category_2')['base_score'].rank(
        ascending=False, method='min'
    )
    master['cat_total'] = master.groupby('category_2')['product_code'].transform('nunique')
    master['cat_percentile'] = ((1 - master['cat_rank'] / master['cat_total']) * 100).round(1)

# 성분 개선 여지 (있으면)
if 'score_golden' in master.columns:
    master['golden_headroom'] = 20 - master['score_golden'].fillna(0)

print(f'\n통합 마스터 테이블: {len(master):,}행')
print(f'고유 제품: {master["product_code"].nunique()}')
print(f'고유 브랜드: {master["brand_name"].nunique()}')
print(f'컬럼 수: {len(master.columns)}')
print(f'\nSWOT 분류 분포:')
print(master['swot_label'].value_counts())

In [ ]:
# ── 저장 ──
master.to_csv(DASH_OUT / 'v_brand_dashboard_master.csv', index=False, encoding='utf-8-sig')
print(f'✅ v_brand_dashboard_master.csv 저장 완료 ({len(master):,}행)')
print(f'   경로: {DASH_OUT / "v_brand_dashboard_master.csv"}')
print(f'\n컬럼 목록:')
for i, col in enumerate(master.columns, 1):
    print(f'  {i:2d}. {col}')

## 5. 데이터 검증: 샘플 브랜드 SWOT 확인

In [ ]:
# 리뷰 수 상위 브랜드 확인
top_brands = (
    master.groupby('brand_name')
    .agg(
        products=('product_code', 'nunique'),
        total_reviews=('total_reviews', 'first'),
        sl_count=('final_soft_landing', lambda x: x.sum() // 8)  # 8 aspect per product
    )
    .sort_values('products', ascending=False)
    .head(15)
)
print('리뷰 수 상위 15개 브랜드:')
print(top_brands)

In [ ]:
def show_product_swot(product_code):
    """특정 제품의 SWOT 요약 출력"""
    prod = master[master['product_code'] == product_code].copy()
    if prod.empty:
        print(f'제품 {product_code} 없음')
        return
    
    info = prod.iloc[0]
    print(f'\n{"="*60}')
    print(f'제품: {info["name"]} ({info["brand_name"]})')
    print(f'가격: {info["price"]:,}원 | 카테고리: {info["category_2"]}')
    print(f'리뷰: {info["total_reviews"]}건 | SL: {info["final_soft_landing"]} (prob: {info["ml_prob"]:.2f})')
    if 'base_score' in info.index and pd.notna(info['base_score']):
        print(f'Base Score: {info["base_score"]} | 카테고리 순위: {int(info["cat_rank"])}/{int(info["cat_total"])}')
    print(f'{"="*60}')
    
    # 강점
    strengths = prod[prod['swot_label'] == 'Strength'].sort_values('gap_pos', ascending=False)
    print(f'\n🟢 강점 (S): {len(strengths)}개')
    for _, r in strengths.iterrows():
        print(f'   {r["aspect"]:10s} | 긍정 {r["pos_pct"]:5.1f}% (평균 대비 +{r["gap_pos"]:4.1f}%p) | 언급률 {r["mention_rate"]:4.1f}%')
    
    # 약점
    weaknesses = prod[prod['swot_label'] == 'Weakness'].sort_values('gap_neg', ascending=False)
    print(f'\n🟠 약점 (W): {len(weaknesses)}개')
    for _, r in weaknesses.iterrows():
        print(f'   {r["aspect"]:10s} | 부정 {r["neg_pct"]:5.1f}% (평균 대비 +{r["gap_neg"]:4.1f}%p) | 언급률 {r["mention_rate"]:4.1f}%')
    
    # 기회
    print(f'\n🔵 기회 (O):')
    if not info['final_soft_landing']:
        print(f'   SL 전환 가능성: {info["ml_prob"]*100:.1f}%')
    else:
        print(f'   이미 연착륙 제품 ✓')
    if 'golden_headroom' in info.index and pd.notna(info['golden_headroom']):
        print(f'   골든 성분 개선 여지: +{info["golden_headroom"]:.0f}점')
    
    # 위협
    print(f'\n🔴 위협 (T):')
    weak_aspects = len(prod[prod['gap_pos'] < -5])
    print(f'   경쟁 열위 Aspect: {weak_aspects}개')
    if 'cat_rank' in info.index and pd.notna(info['cat_rank']):
        print(f'   카테고리 내 순위: {int(info["cat_rank"])}/{int(info["cat_total"])}')

# 리뷰 많은 제품 3개로 테스트
top_products = (
    product_profile
    .sort_values('review_count', ascending=False)
    .head(3)['product_code'].tolist()
)

for pc in top_products:
    show_product_swot(pc)

## 6. 히트맵용 Wide 데이터 (제품 × Aspect)

Tableau 히트맵에서 바로 사용할 수 있도록 피벗 테이블 생성

In [ ]:
# 히트맵용: Long format 그대로 (Tableau에서 Long이 더 편리)
# 선택 카테고리 내 전 제품의 aspect별 pos_pct

heatmap_data = master[[
    'product_code', 'name', 'brand_name', 'category_2', 'price',
    'aspect', 'pos_pct', 'neg_pct', 'mention_rate', 'mention_count',
    'cat_avg_pos_pct', 'gap_pos', 'swot_label',
    'review_count', 'final_soft_landing'
]].copy()

# base_score가 있으면 추가
if 'base_score' in master.columns:
    heatmap_data['base_score'] = master['base_score']
    heatmap_data['cat_rank'] = master['cat_rank']

heatmap_data.to_csv(DASH_OUT / 'v_brand_heatmap_data.csv', index=False, encoding='utf-8-sig')
print(f'✅ v_brand_heatmap_data.csv 저장 완료 ({len(heatmap_data):,}행)')

# Wide format (참고용)
heatmap_wide = product_aspect.pivot_table(
    index='product_code', columns='aspect', values='pos_pct'
).round(1)
print(f'\n히트맵 Wide format: {heatmap_wide.shape}')
heatmap_wide.head()

## 7. 최종 데이터 요약

In [ ]:
print('=' * 60)
print('브랜드 자가진단 대시보드 - 데이터 준비 완료')
print('=' * 60)
print(f'\n출력 파일:')
for f in sorted(DASH_OUT.glob('v_brand_*.csv')):
    size = f.stat().st_size / 1024
    print(f'  {f.name:45s} {size:8.1f} KB')

print(f'\n통계:')
print(f'  총 제품 수: {master["product_code"].nunique()}')
print(f'  총 브랜드 수: {master["brand_name"].nunique()}')
print(f'  카테고리 수: {master["category_2"].nunique()}')
print(f'  Aspect 수: {master["aspect"].nunique()}')
print(f'  마스터 테이블 행: {len(master):,}')
print(f'\nSWOT 분류:')
for label, count in master['swot_label'].value_counts().items():
    print(f'  {label:15s}: {count:,}건 ({count/len(master)*100:.1f}%)')

print(f'\n다음 단계: Tableau에서 v_brand_dashboard_master.csv를 데이터 소스로 추가')